# 4. Full Fine-tuning: Gazeta Summarization

Все 494M параметров обучаются. Float16, gradient checkpointing, 8-bit optimizer.

**Kaggle:** T4 GPU, Internet ON, Input: `gazeta-summaries`

In [1]:
%%capture
!pip install -q rouge-score bitsandbytes

## Загрузка данных и настройка

In [2]:
import os, gc, time, warnings
import torch
import numpy as np
import pandas as pd
from rouge_score import rouge_scorer
warnings.filterwarnings("ignore")

# --- Данные ---
DATA_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "gazeta_train.jsonl" in files:
        DATA_PATH = root
        break
assert DATA_PATH, "Датасет не найден!"

train_df = pd.read_json(os.path.join(DATA_PATH, "gazeta_train.jsonl"), lines=True)
val_df   = pd.read_json(os.path.join(DATA_PATH, "gazeta_val.jsonl"), lines=True)
test_df  = pd.read_json(os.path.join(DATA_PATH, "gazeta_test.jsonl"), lines=True)
print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

# --- Метрики ---
_scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
def evaluate_rouge(preds, refs):
    scores = {"rouge1": [], "rouge2": [], "rougeL": []}
    for p, r in zip(preds, refs):
        s = _scorer.score(r, p)
        for k in scores: scores[k].append(s[k].fmeasure)
    return {k: float(np.mean(v)) for k, v in scores.items()}
def print_rouge(name, sc):
    print(f"{name:<25} R1={sc['rouge1']:.4f}  R2={sc['rouge2']:.4f}  RL={sc['rougeL']:.4f}")

# --- Константы ---
SEED = 42
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
TRAIN_N, VAL_N, TEST_EVAL_N = 2000, 400, 200
MAX_CHARS, MAX_SEQ = 500, 512
PROMPT = "Суммаризируй статью в 2-3 предложения.\n\nСтатья:\n{text}\n\nРезюме:\n"

np.random.seed(SEED); torch.manual_seed(SEED)
train_sub = train_df.sample(n=TRAIN_N, random_state=SEED).reset_index(drop=True)
val_sub   = val_df.sample(n=VAL_N, random_state=SEED).reset_index(drop=True)
test_sub  = test_df.head(TEST_EVAL_N).reset_index(drop=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Train: 60,964  Val: 6,369  Test: 6,793
GPU: Tesla T4
VRAM: 15.6 GB


## Токенизация

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from datasets import Dataset

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def tokenize_fn(examples):
    ids, masks, labels = [], [], []
    for text, summary in zip(examples["text"], examples["summary"]):
        full = PROMPT.format(text=text[:MAX_CHARS]) + summary + tok.eos_token
        enc = tok(full, truncation=True, max_length=MAX_SEQ, add_special_tokens=False)
        ids.append(enc["input_ids"])
        masks.append(enc["attention_mask"])
        labels.append(enc["input_ids"].copy())
    return {"input_ids": ids, "attention_mask": masks, "labels": labels}

tr_ds = Dataset.from_pandas(train_sub[["text","summary"]]).map(tokenize_fn, batched=True, remove_columns=["text","summary"])
vl_ds = Dataset.from_pandas(val_sub[["text","summary"]]).map(tokenize_fn, batched=True, remove_columns=["text","summary"])

sample = tr_ds[0]
n_labels = sum(1 for l in sample["labels"] if l != -100)
print(f"Пример: {len(sample['input_ids'])} токенов, {n_labels} labels")
assert n_labels > 0, "ОШИБКА: labels пустые!"
print("Данные готовы.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Пример: 371 токенов, 371 labels
Данные готовы.


## Обучение

In [4]:
print("=" * 55)
print("  FULL FINE-TUNING (все 494M параметров)")
print("=" * 55)

t0 = time.time()

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Параметры: {trainable/1e6:.1f}M / {total/1e6:.1f}M (100%)")

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./full_ft",
        num_train_epochs=2,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=5e-5,
        warmup_steps=30,
        logging_steps=25,
        eval_strategy="epoch",
        save_strategy="no",
        fp16=False,   # модель уже float16
        report_to="none",
        optim="adamw_8bit",
        max_grad_norm=1.0,
        seed=SEED,
    ),
    train_dataset=tr_ds,
    eval_dataset=vl_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tok, padding=True),
)
trainer.train()
print(f"\nОбучение: {(time.time()-t0)/60:.1f} мин")

`torch_dtype` is deprecated! Use `dtype` instead!


  FULL FINE-TUNING (все 494M параметров)


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Параметры: 494.0M / 494.0M (100%)


Epoch,Training Loss,Validation Loss
1,4.534455,4.167682
2,3.760268,4.295955



Обучение: 16.5 мин


## Оценка

In [5]:
print(f"Генерация ({TEST_EVAL_N} примеров)...")
model.eval()
model.gradient_checkpointing_disable()
preds = []
for i in range(TEST_EVAL_N):
    prompt = PROMPT.format(text=test_sub["text"].iloc[i][:MAX_CHARS])
    inputs = tok(prompt, return_tensors="pt", truncation=True,
                 max_length=MAX_SEQ - 150).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=150, do_sample=False,
                             repetition_penalty=1.2)
    decoded = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    preds.append(decoded.strip())
    if (i+1) % 50 == 0: print(f"  [{i+1}/{TEST_EVAL_N}]")

refs = test_sub["summary"].iloc[:TEST_EVAL_N].tolist()
full_scores = evaluate_rouge(preds, refs)
print()
print_rouge("Full fine-tuning", full_scores)
print(f"\nREF:  {refs[0][:200]}")
print(f"PRED: {preds[0][:200]}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Генерация (200 примеров)...
  [50/200]
  [100/200]
  [150/200]
  [200/200]

Full fine-tuning          R1=0.0048  R2=0.0000  RL=0.0048

REF:  Протестующие против антикоронавирусных мер немцы скандировали имя российского президента, потому что уважают его. Такое мнение выразил депутат городской палаты представителей Гуннар Линдеманн. На этих
PRED: В Е..
